In [1]:
# Import des librairies nécessaires
import pandas as pd

# Chargement des fichiers CSV du dataset Instacart
orders = pd.read_csv('C:/Users/nabem/OneDrive/Belgeler/instacart/orders.csv')
order_products = pd.read_csv('C:/Users/nabem/OneDrive/Belgeler/instacart/order_products__train.csv')
products = pd.read_csv('C:/Users/nabem/OneDrive/Belgeler/instacart/products.csv')

# Vérification des dimensions des datasets
print("orders :", orders.shape)
print("order_products :", order_products.shape)
print("products :", products.shape)

orders : (3421083, 7)
order_products : (1384617, 4)
products : (49688, 4)


In [2]:
# Vérification des colonnes disponibles dans order_products
print(order_products.columns)

Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')


In [3]:
# On travaille sur un échantillon représentatif de 25 000 commandes.
# random_state=42 assure la reproductibilité des résultats.
sample_order_ids = order_products['order_id'].drop_duplicates().sample(25000, random_state = 42)

In [4]:
# Filtrage des lignes correspondant aux 25 000 commandes sélectionnée
sample = order_products[order_products['order_id'].isin(sample_order_ids)]

In [5]:
print("Taille échantillon" sample.shape)

Taille échantillon (263912, 4)


In [6]:
# Fusion avec le fichier products pour obtenir les noms de produits
sample = pd.merge(sample, products, on='product_id')

In [7]:
print(sample.shape)

(263912, 7)


In [8]:
print(sample.head())

   order_id  product_id  add_to_cart_order  reordered  \
0         1       49302                  1          1   
1         1       11109                  2          1   
2         1       10246                  3          0   
3         1       49683                  4          0   
4         1       43633                  5          1   

                                    product_name  aisle_id  department_id  
0                               Bulgarian Yogurt       120             16  
1  Organic 4% Milk Fat Whole Milk Cottage Cheese       108             16  
2                          Organic Celery Hearts        83              4  
3                                 Cucumber Kirby        83              4  
4           Lightly Smoked Sardines in Olive Oil        95             15  


In [9]:
# Sélection des colonnes utiles pour ECLAT
# On garde uniquement order_id et product_name, les autres colonnes ne sont pas nécessaires
sample_eclat = sample[['order_id', 'product_name']]

In [10]:
transactions = sample_eclat.groupby('order_id')['product_name'].apply(list).tolist()

In [11]:
print(f"Nombre de transactions : {len(transactions)}")
print(f"Exemple de transactions : {transactions[0]}")

Nombre de transactions : 25000
Exemple de transaction : ['Bulgarian Yogurt', 'Organic 4% Milk Fat Whole Milk Cottage Cheese', 'Organic Celery Hearts', 'Cucumber Kirby', 'Lightly Smoked Sardines in Olive Oil', 'Bag of Organic Bananas', 'Organic Hass Avocado', 'Organic Whole String Cheese']


In [12]:
from pyECLAT import ECLAT

In [14]:
transactions_df = pd.DataFrame(transactions)
print(transactions_df.shape)

(25000, 72)


In [15]:
eclat_modele = ECLAT(data=transactions_df, verbose=True)

rule_indices, rule_supports = eclat_modele.fit(
    min_support=0.02,    
    min_combination=1,    
    max_combination=3     
)

100%|██████████████████████████████████████████████████████████████████████████| 24696/24696 [00:14<00:00, 1664.42it/s]


Combination 1 by 1


39it [03:53,  5.99s/it]


Combination 2 by 2


741it [34:14,  2.77s/it]


Combination 3 by 3


9139it [7:03:16,  2.78s/it]


In [37]:
from itertools import combinations

resultats_multi = []
items = list(rule_indices.keys())
tidsets = list(rule_indices.values())

# Paires
for i in range(len(items)):
    for j in range(i+1, len(items)):
        intersection = set(tidsets[i]) & set(tidsets[j])
        support = len(intersection) / 25000
        if support >= 0.01:
            resultats_multi.append({
                'itemset': f"{items[i]} & {items[j]}",
                'support': round(support, 4),
                'taille': 2
            })

# Triplets
for i in range(len(items)):
    for j in range(i+1, len(items)):
        for k in range(j+1, len(items)):
            intersection = set(tidsets[i]) & set(tidsets[j]) & set(tidsets[k])
            support = len(intersection) / 25000
            if support >= 0.01:
                resultats_multi.append({
                    'itemset': f"{items[i]} & {items[j]} & {items[k]}",
                    'support': round(support, 4),
                    'taille': 3
                })

df_multi = pd.DataFrame(resultats_multi).sort_values('support', ascending=False)
print(df_multi)

                                              itemset  support  taille
17  Bag of Organic Bananas & Organic Strawberries ...   0.0222       3
15  Organic Strawberries & Bag of Organic Bananas ...   0.0222       2
13  Bag of Organic Bananas & Bag of Organic Banana...   0.0222       2
11      Bag of Organic Bananas & Organic Strawberries   0.0222       2
9       Organic Hass Avocado & Bag of Organic Bananas   0.0184       2
2       Organic Baby Spinach & Bag of Organic Bananas   0.0177       2
4                               Banana & Strawberries   0.0168       2
8                                Banana & Large Lemon   0.0167       2
5                            Banana & Organic Avocado   0.0163       2
6                       Banana & Organic Strawberries   0.0154       2
1                       Organic Baby Spinach & Banana   0.0148       2
12       Bag of Organic Bananas & Organic Raspberries   0.0139       2
16                                Limes & Large Lemon   0.0122       2
14    

In [46]:
regles = []
for _, row in df_multi[df_multi['taille'] == 2].iterrows():
    items_list = row['itemset'].split(' & ')
    item_A = items_list[0]
    item_B = items_list[1]
    
    support_A = rule_supports.get(item_A, None)
    support_B = rule_supports.get(item_B, None)
    
    if support_A and support_B:
        confidence_AB = row['support'] / support_A
        confidence_BA = row['support'] / support_B
        
        regles.append({'regle': f"{item_A} -> {item_B}", 'support': row['support'], 'confidence': round(confidence_AB, 4)})
        regles.append({'regle': f"{item_B} -> {item_A}", 'support': row['support'], 'confidence': round(confidence_BA, 4)})

df_regles = pd.DataFrame(regles).drop_duplicates(subset='regle').sort_values('confidence', ascending=False)
print(df_regles)

                                               regle  support  confidence
30                        Honeycrisp Apple -> Banana   0.0102      0.3643
6     Organic Hass Avocado -> Bag of Organic Bananas   0.0184      0.3353
21     Organic Raspberries -> Bag of Organic Bananas   0.0139      0.3322
11                            Strawberries -> Banana   0.0168      0.3241
15                         Organic Avocado -> Banana   0.0163      0.3071
25       Organic Raspberries -> Organic Strawberries   0.0122      0.2916
0     Organic Strawberries -> Bag of Organic Bananas   0.0222      0.2788
13                             Large Lemon -> Banana   0.0167      0.2697
22                              Limes -> Large Lemon   0.0122      0.2690
8     Organic Baby Spinach -> Bag of Organic Bananas   0.0177      0.2385
33                                   Limes -> Banana   0.0102      0.2249
26      Organic Hass Avocado -> Organic Strawberries   0.0115      0.2095
18                    Organic Baby Spi

In [47]:
df_regles = df_regles.drop_duplicates(subset='regle').sort_values('confidence', ascending=False)
print(df_regles)

                                               regle  support  confidence
30                        Honeycrisp Apple -> Banana   0.0102      0.3643
6     Organic Hass Avocado -> Bag of Organic Bananas   0.0184      0.3353
21     Organic Raspberries -> Bag of Organic Bananas   0.0139      0.3322
11                            Strawberries -> Banana   0.0168      0.3241
15                         Organic Avocado -> Banana   0.0163      0.3071
25       Organic Raspberries -> Organic Strawberries   0.0122      0.2916
0     Organic Strawberries -> Bag of Organic Bananas   0.0222      0.2788
13                             Large Lemon -> Banana   0.0167      0.2697
22                              Limes -> Large Lemon   0.0122      0.2690
8     Organic Baby Spinach -> Bag of Organic Bananas   0.0177      0.2385
33                                   Limes -> Banana   0.0102      0.2249
26      Organic Hass Avocado -> Organic Strawberries   0.0115      0.2095
18                    Organic Baby Spi

In [48]:
df_regles.to_csv('regles_association_eclat.csv', index=False)
print("Sauvegardées !")

Ruled Saved !


In [49]:
import os
print(os.getcwd())

C:\Users\nabem


In [50]:

# items seuls
top_items = pd.DataFrame({
    'support': list(rule_supports.values()),
    'itemsets': [f"frozenset({{'{k}'}})" for k in rule_supports.keys()]
}).sort_values('support', ascending=False)
top_items.to_csv('top_frequent_items.csv', index=False)

# paires et triplés
df_multi.sort_values('support', ascending=False).to_csv('top_itemsets.csv', index=False)

# règles d'associations
df_regles.sort_values('confidence', ascending=False).to_csv('top_rules.csv', index=False)

print("fini")

done
